# 03 · Evaluation and failure-case analysis

**Question:** Does the baseline behave consistently across different parts of the same participant's recording? Where is it wrong?

Compare two **preselected** segments; report both positive and negative findings. These selections are not chosen using reference labels to improve performance, and results should not be generalized to the full dataset.

> This is a diagnostic comparison, not a validated subject-wise test. The code distinguishes synthetic smoke-test mode from genuine UCI data.

### 1. Environment and dataset configuration

**Default: real UCI data.** Set `PPG_DATA_ROOT` to the directory containing `S1/`, `S2/`, etc. Run from the project root (`jupyter lab`). If you cannot access the archive yet, `PPG_DEMO_MODE=1` enables **synthetic engineering-test data only**. Synthetic output must never be reported as dataset findings.

**Security:** loading the official `S1.pkl` deserializes Python pickle, which can run arbitrary code. Inspect the download and use an isolated, trusted environment. The agent cannot choose a data file or filesystem path.

In [ ]:
import os, sys
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython import get_ipython
get_ipython().run_line_magic('matplotlib', 'inline')

HERE = Path.cwd().resolve()
PROJECT_ROOT = HERE if (HERE / 'ppg_demo').is_dir() else HERE.parent
assert (PROJECT_ROOT / 'ppg_demo').is_dir(), 'Run Jupyter from the project root or notebooks/.'
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

DATA_ROOT = Path(os.getenv('PPG_DATA_ROOT', str(PROJECT_ROOT / 'data' / 'PPG_FieldStudy'))).expanduser()
DEMO_MODE = os.getenv('PPG_DEMO_MODE', '0').lower() in ('1', 'true', 'yes')
SUBJECT = os.getenv('PPG_SUBJECT', 'S1')

from ppg_demo.analysis import (load_subject, validate_subject, PPG_HZ, ACC_HZ,
    WIN_SECONDS, STEP_SECONDS, MIN_HZ, MAX_HZ)
from ppg_demo.synthetic import synthetic_record

validate_subject(SUBJECT)
if DEMO_MODE:
    record = synthetic_record(seconds=1100, subject=SUBJECT, bpm=90, movement_from_seconds=600)
    print('⚠ SYNTHETIC SMOKE TEST — NOT PPG-DaLiA MEASUREMENTS')
else:
    dataset_file = DATA_ROOT / SUBJECT / f'{SUBJECT}.pkl'
    if not dataset_file.is_file():
        raise FileNotFoundError(f'Cannot locate {dataset_file}. Extract the official UCI archive and set PPG_DATA_ROOT.')
    record = load_subject(str(dataset_file.resolve()))
    print(f'Official dataset, participant: {SUBJECT} | source: {dataset_file}')
print('BVP shape:', record.bvp.shape, '| ACC shape:', record.acc.shape, '| reference labels:', record.reference.shape)
plt.rcParams.update({'figure.figsize': (11, 4), 'axes.grid': True, 'grid.alpha': .18,
                     'axes.spines.top': False, 'axes.spines.right': False})

## Evaluation design
### 2. Select contrasting recording regions

Use one early and one later segment. Once you inspect the official activity timeline (CSV), you can document the activities associated with these times, but **don't guess activities from clock time**. If you modify these selections, record why.

In [ ]:
from IPython.display import display
from ppg_demo.analysis import analyze_loaded_subject

SEGMENTS = [('early_segment', 30, 120), ('later_segment', 900, 120)]
segment_results = {}
frames = []
metrics = []
for label, start, duration in SEGMENTS:
    assessment = analyze_loaded_subject(record, start, duration, return_trace=True)
    segment_results[label] = assessment
    frame = pd.DataFrame(assessment['windows']).copy()
    frame['segment'] = label
    for method in ['raw', 'filtered']:
        frame[f'{method}_absolute_error_bpm'] = (frame[f'{method}_bpm'] - frame['reference_bpm']).abs()
    frames.append(frame)
    for method, key in [('Raw periodogram', 'raw_periodogram'), ('Filtered + Welch', 'filtered_welch')]:
        metrics.append({'segment': label, 'method': method,
                        **assessment[key],
                        'motion_overlap_percent': assessment['motion_frequency_overlap_percent'],
                        'median_spectral_peak_share': assessment['median_peak_share_heuristic']})
all_windows = pd.concat(frames, ignore_index=True)
metrics_table = pd.DataFrame(metrics)
display(metrics_table.round(2))
print('Mode:', 'SYNTHETIC DEMO — do not report these as real' if DEMO_MODE else 'OFFICIAL UCI DATA')

## Comparisons
### 3. Compare MAE across segments and methods

In [ ]:
p = metrics_table.pivot(index='segment', columns='method', values='mae_bpm')
ax = p.plot(kind='bar', figsize=(10, 4), rot=0)
ax.set(title=f'Segment-level MAE by approach — {"SYNTHETIC" if DEMO_MODE else "dataset"}',
       xlabel='Preselected segment', ylabel='Mean absolute error (BPM)')
ax.legend(title='Method')
plt.tight_layout()
plt.show()

### 4. Inspect the distribution of filtered-estimator absolute errors

A segment mean can hide rare large failures. Always inspect per-window errors.

In [ ]:
fig, ax = plt.subplots()
for segment_name, frame in all_windows.groupby('segment', sort=False):
    valid = frame['filtered_absolute_error_bpm'].dropna()
    ax.hist(valid, bins=16, alpha=.5, label=f'{segment_name} (n={len(valid)})')
ax.set(title=f'Per-window absolute HR errors — {"SYNTHETIC" if DEMO_MODE else "dataset"}',
       xlabel='Absolute error (BPM)', ylabel='Number of overlapping windows')
ax.legend()
plt.show()

### 5. Motion context versus absolute error

ACC motion RMS is a **relative, uncalibrated** indicator, not a calibrated movement score. Spectral overlap is a heuristic, not proof of interference. A visible association should not be interpreted as causality.

In [ ]:
fig, ax = plt.subplots()
for segment_name, frame in all_windows.groupby('segment', sort=False):
    ax.scatter(frame['motion_rms_relative_units'], frame['filtered_absolute_error_bpm'],
               s=18, alpha=.7, label=segment_name)
ax.set(title=f'Motion indicator vs filtered HR error — {"SYNTHETIC" if DEMO_MODE else "dataset"}',
       xlabel='Wrist ACC RMS (relative device units)', ylabel='Absolute HR error (BPM)')
ax.legend()
plt.show()

### 6. Inspect worst windows and describe failure modes

Choose windows for explanation **after** evaluating them; do not retroactively retune the estimator based on the same evaluation windows and report that as unbiased performance.

In [ ]:
worst = (all_windows.dropna(subset=['filtered_absolute_error_bpm'])
         .sort_values('filtered_absolute_error_bpm', ascending=False)
         .loc[:, ['segment', 'start_seconds', 'filtered_bpm', 'reference_bpm',
                  'filtered_absolute_error_bpm', 'motion_rms_relative_units',
                  'motion_frequency_overlap']].head(10))
display(worst.round(2))

## Integrity checks
### 7. Verify the ECG reference is **not** part of the estimator

The test suite also changes reference labels while keeping PPG fixed and ensures the predictions do not change. That invariant is crucial to the assignment.

In [ ]:
from ppg_demo.analysis import estimate_hr_from_ppg
probe = record.bvp[:8*PPG_HZ]
hr_value, spectral_share, peak_hz = estimate_hr_from_ppg(probe)
print('Estimator signature: estimate_hr_from_ppg(ppg_window, prefiltered=False)')
print('Independent estimate (BPM):', hr_value)
print('Reference was NOT supplied to the estimator.')

## Takeaways after real execution

- Add a short paragraph about **which segments were compared** and any known activities from the official activity file.
- Record actual MAE/bias, worst errors, and any relationship to ACC context. Include counterexamples.
- Limitations: two hand-selected sections, overlapping windows, one subject, no calibration of the warning heuristics, no guarantee of clinical-grade accuracy.
- Future work: more participants, subject-wise holdout evaluation, tuned preprocessing using separate development subjects, and artifact-robust methods.

**Next →** `04_ai_agent.ipynb` turns the analysis into one auditable agent tool.